In [1]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import json
import joblib
import pandas as pd
import time
from sklearn.model_selection import GridSearchCV

#### Load the data

In [2]:
df = pd.read_csv('loan-data.csv')

In [4]:
df.head(5)

,age,income,experience,monthly_transactions,account_balance,spending_score,credit_score,default
0,52,26867,27,21,38202,5,673,0
1,30,20841,15,20,28788,8,653,0
2,38,28540,13,28,38626,26,689,0
3,38,40506,17,17,57966,1,669,0
4,31,20057,7,15,27335,1,704,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   age                   3500 non-null   int64
 1   income                3500 non-null   int64
 2   experience            3500 non-null   int64
 3   monthly_transactions  3500 non-null   int64
 4   account_balance       3500 non-null   int64
 5   spending_score        3500 non-null   int64
 6   credit_score          3500 non-null   int64
 7   default               3500 non-null   int64
dtypes: int64(8)
memory usage: 218.9 KB


In [7]:
df.describe()

,age,income,experience,monthly_transactions,account_balance,spending_score,credit_score,default
count,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000
mean,37.347429,24082.561143,15.652000,20.103143,33731.209429,21.887429,671.831429,0.356857
std,11.523942,10042.650610,11.166901,4.428266,14077.596209,26.154681,42.077965,0.479141
min,18.000000,6023.000000,0.000000,7.000000,8484.000000,1.000000,491.000000,0.000000
25%,29.000000,17009.750000,6.000000,17.000000,23729.000000,1.000000,643.000000,0.000000
50%,37.000000,22213.000000,15.000000,20.000000,31122.500000,10.000000,672.000000,0.000000
75%,45.000000,28899.500000,24.000000,23.000000,40454.250000,38.000000,700.000000,1.000000
max,80.000000,105925.000000,40.000000,37.000000,145007.000000,100.000000,829.000000,1.000000


In [8]:
df.isnull().sum()

age                     0
income                  0
experience              0
monthly_transactions    0
account_balance         0
spending_score          0
credit_score            0
default                 0
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

#### Separate feature and target variables

In [11]:
X = df.drop(columns=['default'])

y = df['default']

we first need a baseline model to understand how the model performs with default settings.

#### Train baseline model

In [12]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# Baseline model
model = RandomForestClassifier(random_state=0)
model.fit(X_train, y_train)

# Predictions and accuracy
y_pred = model.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred)

print("Baseline Accuracy:", baseline_acc)

Baseline Accuracy: 0.6028571428571429


We trained our baseline model using `train_test_split` and got `0.6` accuracy for random state of `42`.

To see whether the model’s accuracy is consistent across different data splits, we now repeat the same training process while changing only the random seed.
The following code checks how much the accuracy varies across these splits.

In [13]:
seeds = [1, 5, 21, 42, 77]
accuracies = []

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    model = RandomForestClassifier(random_state=0)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    
    print(f"Random Seed {seed} → Accuracy: {acc:.4f}")

print("\nAccuracy Variation:", accuracies)
print("Min:", np.min(accuracies), "| Max:", np.max(accuracies))

Random Seed 1 → Accuracy: 0.5971
Random Seed 5 → Accuracy: 0.6257
Random Seed 21 → Accuracy: 0.6129
Random Seed 42 → Accuracy: 0.6029
Random Seed 77 → Accuracy: 0.5986

Accuracy Variation: [0.5971428571428572, 0.6257142857142857, 0.6128571428571429, 0.6028571428571429, 0.5985714285714285]
Min: 0.5971428571428572 | Max: 0.6257142857142857


The results show that the model’s accuracy changes noticeably with different random seeds (from 0.59 to 0.62).
This proves that a single train/test split gives unstable and unreliable performance estimates because the score depends on how the data was split.
To get a more consistent evaluation, we now need to switch to cross-validation.

#### 1. Fixing the Unreliable Baseline Using Cross-Validation

In [15]:
model = RandomForestClassifier(random_state = 42)

scores = cross_val_score(model, X, y, cv=5)

print("Cross-Validation Accuracy Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Std Dev:", scores.std())

Cross-Validation Accuracy Scores: [0.61428571 0.61571429 0.61285714 0.62714286 0.62571429]
Mean Accuracy: 0.6191428571428572
Std Dev: 0.006033917737697929


#### 2. Exhaustive Search with GridSearchCV

In [ ]:
# The below code cells takes a while to run. Be patient!

In [24]:
param_grid = {
    'n_estimators': [50, 75, 100],      
    'max_depth': [None, 5, 10],     
    'min_samples_split': [2, 4]    
}

model = RandomForestClassifier(random_state=42)

# Run GridSearch and time it
start = time.time()
grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid.fit(X, y)
end = time.time()
print(f"GridSearch runtime: {end - start:.2f} seconds")

grid_best_score = grid.best_score_
grid_best_params = grid.best_params_

print("Grid Best Params:", grid_best_params)
print("Grid Best CV Score:", grid_best_score)

# compare model performance with basline accuracy
accuracy_diff = grid_best_score - baseline_acc
print(f"Difference in Accuracy is {accuracy_diff}")


GridSearch runtime: 44.92 seconds
Grid Best Params: {'max_depth': 5, 'min_samples_split': 4, 'n_estimators': 50}
Grid Best CV Score: 0.6428571428571428
Difference in Accuracy is 0.039999999999999925


#### 3. RandomizedSearchCV: Smarter Tuning Without the Waiting

In [26]:
# The below code cells takes a while to run. Be patient!

In [27]:
param_dist = {
    'n_estimators': randint(50, 100),
    'max_depth': randint(3, 6),
    'min_samples_split': randint(2, 4)
}

model = RandomForestClassifier(random_state=42)

# Time the RandomizedSearch
start = time.time()

search = RandomizedSearchCV(
                            model,
                            param_distributions=param_dist, n_iter=25,
                            cv=5,
                            scoring='accuracy',
                            n_jobs=-1,
                            random_state=42
)

search.fit(X, y)

end = time.time()
print(f"RandomSearch runtime: {end - start:.2f} seconds")

print("Best Params:", search.best_params_)
print("Best Score:", search.best_score_)

RandomSearch runtime: 31.10 seconds
Best Params: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 88}
Best Score: 0.6431428571428571


#### 4. Which of the following BEST explains why a single train/test split is unreliable for hyperparameter tuning?

In [ ]:
# train / test does not test the model on multiple data variation

#### 5. Which search method is most efficient for tuning 6 hyperparameters with wide ranges?

In [ ]:
# RandomizedSearchCV